In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "leakfix" / "recipe.py").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from leakfix import core, recipe, detect, gemini, compare, residual

if not os.environ.get("GEMINI_API_KEY"):
    zshrc = Path.home() / ".zshrc"
    for line in (zshrc.read_text().splitlines() if zshrc.exists() else []):
        if line.strip().startswith("export GEMINI_API_KEY="):
            os.environ["GEMINI_API_KEY"] = line.split("=", 1)[1].strip().strip("'\"")

print("repo root:", ROOT)
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))

In [ ]:
IMAGE_PATH = "samples/naples_stadium_crowd_01.jpg"
CONFIDENCE = 0.25
MANUAL_BOXES = None

WEIGHTS = "models/leak_detector.pt"
MODELS = {"NB Pro": "gemini-3-pro-image", "NB 2.1": "gemini-nano-banana-2.1"}
PROMPT = gemini.DEFAULT_PROMPT
IMAGE_SIZE = "2K"
OUT_DIR = Path("work/end_to_end") / Path(IMAGE_PATH).stem
OUT_DIR.mkdir(parents=True, exist_ok=True)

original = core.load(IMAGE_PATH)
print(f"{Path(IMAGE_PATH).name}: {original.shape[1]} x {original.shape[0]} px")

In [ ]:
def draw_boxes(ax, boxes, colour="white", labels=None):
    for i, (x0, x1, y0, y1) in enumerate(boxes):
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ls="--", lw=1.5, ec=colour))
        if labels: ax.text(x0 + 10, y0 + 40, labels[i], color=colour, fontsize=9, backgroundcolor=(0, 0, 0, 0.4))

if MANUAL_BOXES:
    boxes, confidences = [tuple(b) for b in MANUAL_BOXES], None
    print(f"using {len(boxes)} manual box(es)")
else:
    detector = detect.load_detector(WEIGHTS)
    found = detect.propose_rgb(detector, original, conf=CONFIDENCE)
    boxes, confidences = [b for b, _ in found], [c for _, c in found]
    print(f"detector found {len(boxes)} leak box(es)" + (": " + ", ".join(f"{c:.2f}" for c in confidences) if found else ""))

fig, ax = plt.subplots(figsize=(8, 8 * original.shape[0] / original.shape[1]))
ax.imshow(original); ax.set_axis_off(); ax.set_title("Detected leak boxes")
draw_boxes(ax, boxes, labels=[f"{c:.2f}" for c in confidences] if confidences else None)
plt.show()

if not boxes:
    raise SystemExit("No leak found. Lower CONFIDENCE, or set MANUAL_BOXES, and run this cell again.")

In [ ]:
inhouse, info = recipe.run_recipe(original, boxes)
print(recipe.describe(info))
inhouse_path = OUT_DIR / "in-house.jpg"
core.save(str(inhouse_path), inhouse)

fig, axes = plt.subplots(1, 2, figsize=(14, 7 * original.shape[0] / original.shape[1]))
for ax, image, title in zip(axes, (original, inhouse), ("Original", "In-house")):
    ax.imshow(np.clip(image, 0, 1)); ax.set_title(title); ax.set_axis_off(); draw_boxes(ax, boxes)
plt.tight_layout(); plt.show()

In [ ]:
client = gemini.make_client()
config = gemini.request_config(IMAGE_SIZE)

def gemini_edit(model_id, input_path, out_path):
    if out_path.exists(): return core.load(str(out_path))
    response = gemini.generate_with_retry(client, model_id, gemini.request_contents(str(input_path), PROMPT), config)
    data = gemini.response_image_bytes(response)
    if data is None:
        raise RuntimeError(f"{model_id} returned no image: {gemini.finish_reason(response)} {gemini.response_text(response)[:200]}")
    out_path.write_bytes(gemini.png_bytes(data))
    return core.load(str(out_path))

results = {}
for name, model_id in MODELS.items():
    for label, source in ((f"{name} alone", Path(IMAGE_PATH)), (f"In-house + {name}", inhouse_path)):
        start = time.time()
        results[label] = gemini_edit(model_id, source, OUT_DIR / f"{label.replace(' ', '_').replace('+', 'plus')}.png")
        print(f"{label:22} done ({time.time() - start:.0f} s)")

In [ ]:
def leak_level(rgb):
    areas = [(b[1] - b[0]) * (b[3] - b[2]) for b in boxes]
    return sum(residual.residual(rgb, b)["warm_mean"] * a for b, a in zip(boxes, areas)) / sum(areas)

original_leak = leak_level(original)
aligned, rows = {}, []
for label, image in [("In-house alone", inhouse)] + list(results.items()):
    if label == "In-house alone": aligned[label], covered = image, None
    else: aligned[label], _, covered = compare.align_with_coverage(original, image, label)
    de76 = compare.measure_collateral(original, aligned[label], boxes, covered=covered)["de76_med"]
    left = 100 * leak_level(aligned[label]) / original_leak if original_leak > 0 else float("nan")
    rows.append((label, de76, left))

print(f"{'Version':22} {'ΔE76 outside':>13} {'Leak left':>10}")
for label, de76, left in rows: print(f"{label:22} {de76:13.2f} {left:9.1f}%")

In [ ]:
four = list(results)
x0, x1, y0, y1 = max(boxes, key=lambda b: (b[1] - b[0]) * (b[3] - b[2]))
side = min(700, *original.shape[:2])
cx = int(np.clip((x0 + x1) // 2 - side // 2, 0, original.shape[1] - side))
cy = int(np.clip((y0 + y1) // 2 - side // 2, 0, original.shape[0] - side))
scores = {label: (de76, left) for label, de76, left in rows}

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for i, label in enumerate(["Original"] + four):
    image = original if label == "Original" else aligned[label]
    axes[0, i].imshow(np.clip(image, 0, 1)); axes[0, i].set_title(label, fontsize=11)
    draw_boxes(axes[0, i], boxes); axes[0, i].add_patch(plt.Rectangle((cx, cy), side, side, fill=False, lw=1.5, ec="#ffd400"))
    axes[1, i].imshow(np.clip(image[cy:cy + side, cx:cx + side], 0, 1))
    if label != "Original": axes[1, i].set_title(f"ΔE76 {scores[label][0]:.1f} · leak left {scores[label][1]:.0f}%", fontsize=10)
    for ax in axes[:, i]: ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT_DIR / "four_results.jpg", dpi=110)
plt.show()
print("saved to", OUT_DIR)